In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

#Library

In [ ]:
pip install transformers==4.3.0

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 1.8 MB 32.8 MB/s 
     |████████████████████████████████| 880 kB 14.7 MB/s 
     |████████████████████████████████| 3.3 MB 53.1 MB/s 
  Created wheel for sacremoses: filename=sacremoses-0.0.53-py3-none-any.whl size=895260 sha256=ebcaff9ee47e6b3634971e1ebe1d8054f4d1e56549ef7cf22a308e59d335821d
  Stored in directory: /root/.cache/pip/wheels/87/39/dd/a83eeef36d0bf98e7a4d1933a4ad2d660295a40613079bafc9
Successfully built sacremoses


In [ ]:
pip install vncorenlp

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 2.6 MB 28.4 MB/s 
  Created wheel for vncorenlp: filename=vncorenlp-1.0.3-py3-none-any.whl size=2645951 sha256=aa7d1257ea3228a45a33545a3282c7d538bdbf785ae43fbba2d207c485315eb8
  Stored in directory: /root/.cache/pip/wheels/0c/d8/f2/d28d97379b4f6479bf51247c8dfd57fa00932fa7a74b6aab29
Successfully built vncorenlp


In [ ]:
!mkdir -p vncorenlp/models/wordsegmenter
!wget https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.1.1.jar
!wget https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/vi-vocab
!wget https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/wordsegmenter.rdr
!mv VnCoreNLP-1.1.1.jar vncorenlp/
!mv vi-vocab vncorenlp/models/wordsegmenter/
!mv wordsegmenter.rdr vncorenlp/models/wordsegmenter/

--2022-06-11 12:54:17--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.1.1.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412575 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.1.1.jar’

VnCoreNLP-1.1.1.jar 100%[===================>]  26.14M  --.-KB/s    in 0.1s    

2022-06-11 12:54:20 (193 MB/s) - ‘VnCoreNLP-1.1.1.jar’ saved [27412575/27412575]

--2022-06-11 12:54:20--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/vi-vocab
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting respon

#Data loader and preprocessing

In [ ]:
#load dataset crawl được từ youtube
import pandas as pd

test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data4.csv", index_col=False)
test.reset_index(drop=True)
test['Pseudo Label'] = 0
test

,Comment,Likes,Pseudo Label
0,STREET DANCE VIỆT NAM - TẬP 3 full video https...,2,0
1,Mỗi một giây của chương trình đều làm mình nổi...,73,0
2,Phải đến đoạn Trọng Hiếu nhảy mới dám mở to 2 ...,76,0
3,Nhìn Trọng Hiếu nhảy mà tui muốn nhảy theo luôn,13,0
4,"Bật lại mấy cái clip hip hop ngày xưa , giờ co...",14,0
5,Chời ơi đỡ tui :)))))) cừi chết với cú ke đầu ...,45,0
6,"Với khả năng nhảy, đặc biệt là quả ke đầu này,...",76,0
7,Nhìn mặt bà pu hài vl 🤣🤣🤣🤣 t coi trên fb nói v...,13,0
8,"Show rất hay, ngoài làm thí sinh thì họ cũng l...",19,0
9,Một gameshow về dance mà bốn đội trưởng chỉ mỗ...,78,0


In [ ]:
from vncorenlp import VnCoreNLP

vncorenlp = VnCoreNLP("vncorenlp/VnCoreNLP-1.1.1.jar", annotators="wseg", max_heap_size='-Xmx500m')

In [ ]:
#pre-process
import re
import numpy as np

STOPWORDS = '/content/drive/MyDrive/Colab Notebooks/ViHSD/vietnamese-stopwords.txt'
with open(STOPWORDS, "r") as ins:
    stopwords = []
    for line in ins:
        dd = line.strip('\n')
        stopwords.append(dd)
    stopwords = set(stopwords)

def filter_stop_words(train_sentences, stop_words):
    new_sent = [word for word in train_sentences.split() if word not in stop_words]
    train_sentences = ' '.join(new_sent)

    return train_sentences

def deEmojify(text):
    regrex_pattern = re.compile(pattern = "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "]+", flags = re.UNICODE)
    return regrex_pattern.sub(r'',text)

def preprocess(text, tokenized=True, lowercased=True):
    # text = ViTokenizer.tokenize(text)
    # text = ' '.join(vncorenlp.tokenize(text)[0])
    text = filter_stop_words(text, stopwords)
    text = deEmojify(text)
    text = text.lower() if lowercased else text
    if tokenized:
        pre_text = ""
        sentences = vncorenlp.tokenize(text)
        for sentence in sentences:
            pre_text += " ".join(sentence)
        text = pre_text
    return text

def pre_process_features(X, y, tokenized=True, lowercased=True):
    X = [preprocess(str(p), tokenized=tokenized, lowercased=lowercased) for p in list(X)]
    for idx, ele in enumerate(X):
        if not ele:
            np.delete(X, idx)
            np.delete(y, idx)
    return X, y

X_test = test['Comment']
y_test = test['Pseudo Label'].values

#Model

In [ ]:
#load model đã được train
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, BertTokenizer, BertForSequenceClassification

import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt

import torch

model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels = 3)

model.load_state_dict(torch.load('/content/drive/MyDrive/Colab Notebooks/ViHSD/transformer_model/phobert-v3/pytorch_model.bin'))


Downloading:   0%|          | 0.00/557 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of the model checkpoint at vinai/phobert-base were not used when initializing RobertaForSequenceClassification: ['lm_head.bias', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.decoder.weight', 'lm_head.decoder.bias', 'roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['

<All keys matched successfully>

In [ ]:
X_test

0      STREET DANCE VIỆT NAM - TẬP 3 full video https...
1      Mỗi một giây của chương trình đều làm mình nổi...
2      Phải đến đoạn Trọng Hiếu nhảy mới dám mở to 2 ...
3        Nhìn Trọng Hiếu nhảy mà tui muốn nhảy theo luôn
4      Bật lại mấy cái clip hip hop ngày xưa , giờ co...
                             ...                        
173                                  Rạp xiếc trung ương
174                                                  ???
175                                    Tao đang coi clgv
176        t chỉ muốn biết là t đang xem cl gì thế này??
177                         Clmn làm vô coi ke đầu !!!!!
Name: Comment, Length: 178, dtype: object

In [ ]:
test_X, test_y = pre_process_features(X_test, y_test, tokenized=True, lowercased = False)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base",use_fast=False)

import torch

class BuildDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

test_encodings = tokenizer(test_X, truncation=True, padding=True, max_length=100)
test_dataset = BuildDataset(test_encodings, test_y)

Downloading:   0%|          | 0.00/895k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embedding are fine-tuned or trained.


In [ ]:
training_args = TrainingArguments(output_dir='/content/drive/MyDrive/Colab Notebooks/ViHSD/transformer_model/phobert-v3/',no_cuda=False)

trainer = Trainer(model=model,args=training_args)

y_pred_classify = trainer.predict(test_dataset)

In [ ]:
y_pred = np.argmax(y_pred_classify.predictions, axis=-1)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       1, 0])

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option("max_rows", None)
#Đưa vector dự đoán label vào dataframe
df_predict = pd.DataFrame(y_pred)
df_predict.columns = ['predict']
#Xóa cột likes
test = test.drop(columns=['Likes', 'Pseudo Label'])
#Nối cột predcit vào test
df_test_predict = test.join(df_predict)
df_test_predict

,Comment,predict
0,STREET DANCE VIỆT NAM - TẬP 3 full video https...,0
1,Mỗi một giây của chương trình đều làm mình nổi...,0
2,Phải đến đoạn Trọng Hiếu nhảy mới dám mở to 2 ...,0
3,Nhìn Trọng Hiếu nhảy mà tui muốn nhảy theo luôn,0
4,"Bật lại mấy cái clip hip hop ngày xưa , giờ co...",0
5,Chời ơi đỡ tui :)))))) cừi chết với cú ke đầu ...,0
6,"Với khả năng nhảy, đặc biệt là quả ke đầu này,...",0
7,Nhìn mặt bà pu hài vl 🤣🤣🤣🤣 t coi trên fb nói v...,0
8,"Show rất hay, ngoài làm thí sinh thì họ cũng l...",0
9,Một gameshow về dance mà bốn đội trưởng chỉ mỗ...,0
